# Cell-by-cell comparison of Textract output against manual ground truth

Implements the comparison metrics described in Section 5.1 of the paper.
For each cell of every cleaned 1956 weekly table, compares the Textract value
against the manually-entered value and writes a long-format CSV of the result.

**Inputs**
- `data/textract_cleaned/1956/*_tables.csv` &mdash; 52 files
- `data/textract_cleaned/1956/*_confidences.csv` &mdash; 52 files (optional; if missing,
  the `Confidence_Value` column is left empty)
- `data/manual_cleaned/1956/*.csv` &mdash; 52 files

**Outputs (under `data/analysis_intermediates/`)**
- `Levels_of_Accuracy.csv` &mdash; one row per cell across all 52 weekly tables,
  with each of the four equality flags described in Section 5.1.1
- `rows_with_no_equality.csv` &mdash; subset where `Any_Equality == False`, with
  `Lev_Dist` (Levenshtein distance) and `Lev_Sim` (normalized similarity)
  columns added; this is the input to Figs 4 and 5

**Downstream**
- `Levels_of_Accuracy.csv` &rarr; Section 5.1.1 percentages (71% &rarr; 97%)
- `rows_with_no_equality.csv` &rarr; Figs 4, 5 (`analysis/plot_levenshtein.py`)


In [2]:
# Configuration. All paths are relative to the repository root.

import os

REPO_ROOT = ".."   # this notebook lives in postprocessing/

YEAR = 1956
N_WEEKS = 52
FILE_PREFIX = f"cdi_ca_{YEAR}_wk_prov_dbs_Part"

TEXTRACT_DIR  = os.path.join(REPO_ROOT, "data", "textract_cleaned", str(YEAR))
MANUAL_DIR    = os.path.join(REPO_ROOT, "data", "manual_cleaned",   str(YEAR))
OUTPUT_DIR    = os.path.join(REPO_ROOT, "data", "analysis_intermediates")

LEVELS_CSV    = os.path.join(OUTPUT_DIR, "Levels_of_Accuracy.csv")
NO_EQUAL_CSV  = os.path.join(OUTPUT_DIR, "rows_with_no_equality.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
# Equality checks for a pair of (textract, manual) values.

import csv

def is_empty_or_dash(value):
    s = str(value).strip()
    return s == "" or s == "-" or s.replace(".", "") == ""

def normalize_commas_spaces(value):
    return str(value).replace(",", "").replace(" ", "")

def extract_numeric_string(value):
    return "".join(ch for ch in str(value) if ch.isdigit() or ch in "+-.")

def cells_equal(a, b):
    """Raw equality (string comparison)."""
    return a == b

def cells_equal_no_commas_spaces(a, b):
    """Equality after stripping commas and spaces."""
    return normalize_commas_spaces(a) == normalize_commas_spaces(b)

def cells_equal_shifted(row_a, j, b, n_cols):
    """
    Either (1) cell (i, j) of textract matches manual after stripping commas/spaces,
    or (2) cell (i, j+1) of textract matches manual after stripping commas/spaces.
    Captures the common Textract failure of shifting an entire row by one cell.
    """
    a = str(row_a[j]) if j < len(row_a) else ""
    a_norm = normalize_commas_spaces(a)
    b_norm = normalize_commas_spaces(b)
    shifted = False
    if j + 1 < n_cols and j + 1 < len(row_a):
        shifted = normalize_commas_spaces(row_a[j + 1]) == b_norm
    return shifted or (a_norm == b_norm)

def cells_numbers_only_equal(a, b):
    """Equality after extracting only the numeric characters. Returns False if either side is empty/dash."""
    if is_empty_or_dash(a) or is_empty_or_dash(b):
        return False
    na, nb = extract_numeric_string(a), extract_numeric_string(b)
    if not na or not nb:
        return False
    return na == nb


In [4]:
# Build Levels_of_Accuracy.csv: one row per cell across all weekly tables.

import pandas as pd

def read_csv_raw(path):
    """Read a CSV with all values as strings, no NA coercion."""
    return pd.read_csv(path, header=None, dtype=str,
                       keep_default_na=False, na_filter=False)

def compare_pair(textract_table_path, manual_table_path, confidence_path=None):
    """Compare one (textract, manual) pair. Returns list of dicts (one per cell)."""
    df_t = read_csv_raw(textract_table_path)
    df_m = read_csv_raw(manual_table_path)
    df_c = read_csv_raw(confidence_path) if confidence_path and os.path.exists(confidence_path) else None

    n_rows = min(df_t.shape[0], df_m.shape[0])
    n_cols = min(df_t.shape[1], df_m.shape[1])
    if df_c is not None:
        n_rows = min(n_rows, df_c.shape[0])
        n_cols = min(n_cols, df_c.shape[1])

    out = []
    fname = os.path.basename(textract_table_path)
    for i in range(n_rows):
        row_t = df_t.iloc[i].tolist()
        for j in range(n_cols):
            a, b = df_t.iat[i, j], df_m.iat[i, j]
            eq       = cells_equal(a, b)
            eq_nc    = cells_equal_no_commas_spaces(a, b)
            eq_shift = cells_equal_shifted(row_t, j, b, n_cols)
            eq_num   = cells_numbers_only_equal(a, b)
            out.append({
                "file_name": fname,
                "row_index": i, "col_index": j,
                "AWS_Value": a, "Manual_Value": b,
                "Equal": eq,
                "Equal_No_Commas_No_Spaces": eq_nc,
                "Equal_Shifted": eq_shift,
                "Only_Numbers_Equal_Or_Shifted": eq_num or eq_shift,
                "Any_Equality": eq or eq_nc or eq_shift or eq_num,
                "Confidence_Value": df_c.iat[i, j] if df_c is not None else "",
            })
    return out


all_rows = []
for i in range(1, N_WEEKS + 1):
    base = f"{FILE_PREFIX}{i}"
    t_path = os.path.join(TEXTRACT_DIR, f"{base}_tables.csv")
    m_path = os.path.join(MANUAL_DIR,   f"{base}.csv")
    c_path = os.path.join(TEXTRACT_DIR, f"{base}_confidences.csv")
    if not (os.path.exists(t_path) and os.path.exists(m_path)):
        print(f"Skipping week {i} (missing input)")
        continue
    all_rows.extend(compare_pair(t_path, m_path, c_path))

df = pd.DataFrame(all_rows)
df.to_csv(LEVELS_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"Saved {len(df):,} rows to {LEVELS_CSV}")


Saved 53,872 rows to ..\data\analysis_intermediates\Levels_of_Accuracy.csv


In [5]:
# Section 5.1.1 percentages: progressively more lenient equality criteria.
#
# Reports both the population-wide percentages (every cell, including dashes
# for zero cases and dots for missing data) and the percentages restricted
# to cells where both Textract and the manual entry contain numeric values
# (the substantive comparison: of the cells holding actual disease counts,
# how many did Textract read correctly?).

import pandas as pd

df = pd.read_csv(LEVELS_CSV, dtype=str, keep_default_na=False, na_filter=False)

def pct(sub, col):
    return 100.0 * sub[col].astype(str).str.lower().eq("true").mean()

def is_numeric_like(v):
    s = str(v).strip().replace(",", "")
    if s == "":
        return False
    try:
        float(s)
        return True
    except ValueError:
        return False

both_numeric = df["AWS_Value"].apply(is_numeric_like) & df["Manual_Value"].apply(is_numeric_like)
numeric_only = df[both_numeric]

cols = ["Equal", "Equal_No_Commas_No_Spaces",
        "Only_Numbers_Equal_Or_Shifted", "Any_Equality"]
labels = ["Strict equality",
          "+ Commas/spaces normalised",
          "+ Numeric-only equality",
          "+ Row-shift tolerated"]

print(f"All cells (n = {len(df):,}):")
for c, label in zip(cols, labels):
    print(f"  {label:30s}: {pct(df, c):5.2f}%")

print(f"\nCells with numeric data on both sides (n = {len(numeric_only):,}):")
for c, label in zip(cols, labels):
    print(f"  {label:30s}: {pct(numeric_only, c):5.2f}%")


All cells (n = 53,872):
  Strict equality               : 73.98%
  + Commas/spaces normalised    : 76.84%
  + Numeric-only equality       : 88.53%
  + Row-shift tolerated         : 88.53%

Cells with numeric data on both sides (n = 26,069):
  Strict equality               : 77.92%
  + Commas/spaces normalised    : 83.80%
  + Numeric-only equality       : 95.58%
  + Row-shift tolerated         : 95.58%


In [6]:
# Build rows_with_no_equality.csv with Levenshtein columns added.
# This is the input to Figs 4 and 5 in the paper.

import pandas as pd
import nltk

df = pd.read_csv(LEVELS_CSV, dtype=str, keep_default_na=False, na_filter=False)
mask = df["Any_Equality"].astype(str).str.upper() == "FALSE"
bad = df[mask].copy()

# Ensure string types for Levenshtein
bad["AWS_Value"] = bad["AWS_Value"].astype(str)
bad["Manual_Value"] = bad["Manual_Value"].astype(str)

# Edit distance
bad["Lev_Dist"] = [
    nltk.edit_distance(a, b)
    for a, b in zip(bad["AWS_Value"], bad["Manual_Value"])
]

# Normalized similarity: 1 - edit_distance / max(len_a, len_b)
max_len = bad["AWS_Value"].str.len().combine(bad["Manual_Value"].str.len(), max)
max_len = max_len.replace(0, 1)  # avoid division by zero
bad["Lev_Sim"] = 1 - bad["Lev_Dist"] / max_len

bad.to_csv(NO_EQUAL_CSV, index=False)
print(f"Saved {len(bad):,} mismatched rows to {NO_EQUAL_CSV}")
print(f"Mean Levenshtein distance among mismatches: {bad['Lev_Dist'].mean():.3f}")


Saved 6,177 mismatched rows to ..\data\analysis_intermediates\rows_with_no_equality.csv
Mean Levenshtein distance among mismatches: 1.838
